In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.cluster import KMeans
from sklearn.metrics import classification_report

import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("../data/processed/messages_with_sentiment.csv")

df = df[df["comment_clean"].notna()]
df.head()


,id,date,date_unixtime,professor_id,professor_name_clean,department,course_name_clean,rating_1,rating_2,rating_3,rating_4,rating_5,rating_6,grading_status,attendance_status,comment_clean,sentiment
0,26,2021-09-05T02:19:28,1630792168,13,داریم n nلطفا استاد هایی که میخواید معرفی کنید...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,نامشخص,نامشخص,سنجی u200cها بیان کننده u200cی توان علمی اساتی...,neutral
1,55,2021-09-05T15:17:17,1630838837,32,کلاس داشته n بهمن 98 n nتوضیحات n در مجموع سخت...,NaN,5 n نحوه مدیریت کلاس نظم و زمان 8 n پاسخگویی ح...,NaN,NaN,NaN,NaN,NaN,NaN,سختگیر,نامشخص,خودش n nبرای ثبت معرفی استاد به ربات زیر پیام ...,neutral
2,66,2021-09-05T15:43:56,1630840436,40,ی خوب گفتن و همه مخالفن چیو نشون میده nیا برعک...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,آسان,نامشخص,مخالف داره یعنی چی n nمثلا معرفی ریپلای شده رو...,neutral
3,76,2021-09-05T16:12:31,1630842151,48,کلاس داشته n بهمن 99 n nتوضیحات n خودش تدریس ن...,NaN,8 n نحوه مدیریت کلاس نظم و زمان 3 n پاسخگویی ح...,NaN,NaN,NaN,NaN,NaN,NaN,نامشخص,نامشخص,ی n type hashtag text مهندسی_عمران n زمین شناس...,neutral
4,85,2021-09-05T21:17:39,1630860459,57,کلاس داشته n بهمن 99 n nتوضیحات n فقط کافیه تو...,NaN,9 n نحوه مدیریت کلاس نظم و زمان 9 n پاسخگویی ح...,NaN,NaN,NaN,NaN,NaN,NaN,منصفانه,نامشخص,سنجی یا آزمون میذارن و چند ثانیه فرصت داری جوا...,neutral


In [3]:
df_sup = df[df["sentiment"] != "neutral"].copy()
df_sup["sentiment"].value_counts()


sentiment
positive    107
negative     21
Name: count, dtype: int64

In [4]:
X = df_sup["comment_clean"]
y = df_sup["sentiment"]

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_vec = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_vec,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [5]:
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("Logistic Regression Results")
print(classification_report(y_test, y_pred_lr))


Logistic Regression Results
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         4
    positive       0.85      1.00      0.92        22

    accuracy                           0.85        26
   macro avg       0.42      0.50      0.46        26
weighted avg       0.72      0.85      0.78        26



c:\Users\zahra\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\zahra\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\zahra\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [6]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Results")
print(classification_report(y_test, y_pred_rf))


Random Forest Results
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         4
    positive       0.83      0.86      0.84        22

    accuracy                           0.73        26
   macro avg       0.41      0.43      0.42        26
weighted avg       0.70      0.73      0.71        26



In [8]:
df_cluster = df.copy()

X_text = df_cluster["comment_clean"]

vectorizer_cluster = TfidfVectorizer(
    max_features=3000,
    stop_words=None
)

X_cluster_vec = vectorizer_cluster.fit_transform(X_text)


In [9]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df_cluster["cluster"] = kmeans.fit_predict(X_cluster_vec)
df_cluster["cluster"].value_counts()


cluster
2    312
0    158
1     30
Name: count, dtype: int64

In [10]:
terms = vectorizer_cluster.get_feature_names_out()
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for i in range(3):
    print(f"\nCluster {i}:")
    for ind in order_centroids[i, :10]:
        print(terms[ind])



Cluster 0:
mention
type
text
ostad_elmosi
معرفی
استاد
به
اساتید
زیر
دانشگاه

Cluster 1:
10
nمنابع
دانشجو
با
مهم
کلاس
است
تدریس
دانشجویان
پاسخگویی

Cluster 2:
type
text
mention
معرفی
ostad_elmosi
نمره
که
هم
به
رو


In [11]:
pd.crosstab(df_cluster["cluster"], df_cluster["sentiment"], normalize="index")


sentiment,negative,neutral,positive
cluster,,,
0,0.018987,0.848101,0.132911
1,0.000000,0.700000,0.300000
2,0.057692,0.695513,0.246795


In [12]:
df_cluster.to_csv("../data/processed/messages_with_clusters.csv", index=False)
print("✅ messages_with_clusters.csv saved")


✅ messages_with_clusters.csv saved
